# Clone repo and setup

In [1]:
# setup
!git clone https://github.com/ndromanenko/dla_asr.git

Cloning into 'dla_asr'...
remote: Enumerating objects: 540, done.
remote: Counting objects: 100% (540/540), done.
remote: Compressing objects: 100% (258/258), done.
remote: Total 540 (delta 263), reused 522 (delta 246), pack-reused 0 (from 0)
Receiving objects: 100% (540/540), 1.57 MiB | 4.49 MiB/s, done.
Resolving deltas: 100% (263/263), done.


In [2]:
%cd dla_asr

/content/dla_asr


In [3]:
%pip install -q uv
!uv --version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 103.1 MB/s eta 0:00:00
uv 0.13.4 (x86_64-unknown-linux-gnu)


In [4]:
!uv python install 3.12
!uv python list --only-installed

Installed Python 3.12.15 in 1.14s
 + cpython-3.12.15-linux-x86_64-gnu (python3.12)
cpython-3.13.15-linux-x86_64-gnu    /usr/local/bin/python
cpython-3.13.15-linux-x86_64-gnu    /usr/bin/python3.13
cpython-3.13.15-linux-x86_64-gnu    /usr/bin/python3 -> /etc/alternatives/python3
cpython-3.12.15-linux-x86_64-gnu    /root/.local/share/uv/python/cpython-3.12-linux-x86_64-gnu/bin/python3.12
cpython-3.12.3-linux-x86_64-gnu     /usr/bin/python3.12


In [5]:
!uv sync --python 3.12 --locked

Using CPython 3.12.15
Creating virtual environment at: .venv
Resolved 127 packages in 2ms
Prepared 124 packages in 1m 23s
Installed 124 packages in 786ms
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + annotated-types==0.8.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.15.1
 + attrs==26.1.0
 + certifi==2026.7.22
 + cffi==2.1.1
 + cfgv==3.5.0
 + charset-normalizer==3.5.2
 + click==8.5.0
 + comet-ml==3.58.8
 + configobj==5.0.9
 + contourpy==1.4.0
 + cycler==0.12.1
 + datasets==5.1.0
 + dill==0.4.1
 + distlib==0.4.4
 + dulwich==1.2.17
 + everett==3.1.0
 + filelock==4.1.0
 + fonttools==4.66.1
 + frozenlist==1.8.0
 + fsspec==2026.7.0
 + googleapis-common-protos==1.75.5
 + h11==0.16.0
 + hf-xet==1.7.0
 + httpcore2==2.13.1
 + httpx2==2.13.1
 + huggingface-hub==2.2.0
 + hydra-core==1.3.7
 + identify==2.6.20
 + idna==3.20
 + jinja2==3.1.6
 + jsonschema==4.26.0
 + jsonschema-specifications==2025.9.1
 + julius==0.2.8
 + kiwisolver==1.5.1
 + markdown-it-py==4.2.0
 + markupsafe

# Dataset and model

In [6]:
!uv run python download_model.py



Model saved to /content/dla_asr/ckpt/checkpoint_60epoch.pth


In [7]:
!uv pip install gdown

Using Python 3.13.15 environment at: /usr
Checked 1 package in 123ms


In [8]:
DATASET = "https://drive.google.com/file/d/1WoPJZKyRnyRlQcyDDXIm1qsJh4ClRWIX/view?usp=sharing"

!uv run gdown --fuzzy "{DATASET}" -O data_test.zip

Downloading...
From: https://drive.google.com/uc?id=1WoPJZKyRnyRlQcyDDXIm1qsJh4ClRWIX
To: /content/dla_asr/data_test.zip
100% 626k/626k [00:00<00:00, 7.61MB/s]


In [9]:
!unzip -q /content/dla_asr/data_test.zip -d /content/dla_asr/

# Inference or Testing

In [10]:
# inference only argmax
!MPLBACKEND=Agg uv run inference.py -cn inference dataloader.batch_size=1 text_encoder.beam_size=10 metrics=argmax inferencer.decode=argmax

ConformerModel(
  (conv_subsampling): ConvSubsampling(
    (conv1): Conv2d(1, 144, kernel_size=(3, 3), stride=(2, 2))
    (conv2): Conv2d(144, 144, kernel_size=(3, 3), stride=(2, 2))
    (dropout): Dropout(p=0.1, inplace=False)
    (relu_1): ReLU()
    (relu_2): ReLU()
    (flatten): Flatten(start_dim=2, end_dim=-1)
    (linear): Linear(in_features=2736, out_features=144, bias=True)
  )
  (pos_emb): PositionEncoding()
  (blocks): ModuleList(
    (0-15): 16 x ConformerBlock(
      (multihead_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=144, out_features=144, bias=True)
      )
      (ffn_1): FeedForward(
        (layer_norm): LayerNorm((144,), eps=1e-05, elementwise_affine=True)
        (linear_1): Linear(in_features=144, out_features=576, bias=True)
        (non_linear): SiLU()
        (dropout_1): Dropout(p=0.1, inplace=False)
        (linear_2): Linear(in_features=576, out_features=144, bias=True)
        (dropout_2): Dropout(p=0.1, inpla

In [11]:
TARGET_DIR = "test_audio/transcriptions"
PRED_DIR = "data/saved/predictions/test"

!uv run calc_metrics.py --target_dir "{TARGET_DIR}" --pred_dir "{PRED_DIR}"

CER - 0.06158893278140758
WER - 0.18053354053354054


In [12]:
# inference only beamsearch
!MPLBACKEND=Agg uv run inference.py -cn inference dataloader.batch_size=1 text_encoder.beam_size=10 metrics=beam inferencer.decode=beam

ConformerModel(
  (conv_subsampling): ConvSubsampling(
    (conv1): Conv2d(1, 144, kernel_size=(3, 3), stride=(2, 2))
    (conv2): Conv2d(144, 144, kernel_size=(3, 3), stride=(2, 2))
    (dropout): Dropout(p=0.1, inplace=False)
    (relu_1): ReLU()
    (relu_2): ReLU()
    (flatten): Flatten(start_dim=2, end_dim=-1)
    (linear): Linear(in_features=2736, out_features=144, bias=True)
  )
  (pos_emb): PositionEncoding()
  (blocks): ModuleList(
    (0-15): 16 x ConformerBlock(
      (multihead_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=144, out_features=144, bias=True)
      )
      (ffn_1): FeedForward(
        (layer_norm): LayerNorm((144,), eps=1e-05, elementwise_affine=True)
        (linear_1): Linear(in_features=144, out_features=576, bias=True)
        (non_linear): SiLU()
        (dropout_1): Dropout(p=0.1, inplace=False)
        (linear_2): Linear(in_features=576, out_features=144, bias=True)
        (dropout_2): Dropout(p=0.1, inpla

In [13]:
!uv run calc_metrics.py --target_dir "{TARGET_DIR}" --pred_dir "{PRED_DIR}"

CER - 0.05983454681649529
WER - 0.16403522403522403


In [14]:
# inference beamsearch and argmax, but pred save with beam_search
!MPLBACKEND=Agg uv run inference.py -cn inference dataloader.batch_size=1 text_encoder.beam_size=10

ConformerModel(
  (conv_subsampling): ConvSubsampling(
    (conv1): Conv2d(1, 144, kernel_size=(3, 3), stride=(2, 2))
    (conv2): Conv2d(144, 144, kernel_size=(3, 3), stride=(2, 2))
    (dropout): Dropout(p=0.1, inplace=False)
    (relu_1): ReLU()
    (relu_2): ReLU()
    (flatten): Flatten(start_dim=2, end_dim=-1)
    (linear): Linear(in_features=2736, out_features=144, bias=True)
  )
  (pos_emb): PositionEncoding()
  (blocks): ModuleList(
    (0-15): 16 x ConformerBlock(
      (multihead_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=144, out_features=144, bias=True)
      )
      (ffn_1): FeedForward(
        (layer_norm): LayerNorm((144,), eps=1e-05, elementwise_affine=True)
        (linear_1): Linear(in_features=144, out_features=576, bias=True)
        (non_linear): SiLU()
        (dropout_1): Dropout(p=0.1, inplace=False)
        (linear_2): Linear(in_features=576, out_features=144, bias=True)
        (dropout_2): Dropout(p=0.1, inpla

In [15]:
!uv run calc_metrics.py --target_dir "{TARGET_DIR}" --pred_dir "{PRED_DIR}"

CER - 0.05983454681649529
WER - 0.16403522403522403
